# ConsentShield Phase 3 — Fine-tune CLIP (Colab)

Trains `openai/clip-vit-base-patch32` on `datasets/training/vision/`.

**Runtime:** GPU (T4 / A100). Enable: Runtime → Change runtime type → GPU.

In [ ]:
# 1) Mount Drive or upload the repo zip, then set REPO_ROOT
from google.colab import drive
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/ConsentShield'  # <-- edit
%cd {REPO_ROOT}
!pip -q install -U torch torchvision transformers accelerate scikit-learn pillow pyyaml matplotlib

In [ ]:
import os, json, torch
from pathlib import Path
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')
assert Path('datasets/training/vision/train.jsonl').is_file(), 'Upload Phase 3.2 vision JSONL first'
print(json.load(open('datasets/training/vision/stats.json'))['total_usable_images'])

In [ ]:
# Full Phase 3 training (5 epochs, bs=32, AMP on CUDA)
!python -m ai.training.train_clip --epochs 5 --batch-size 32 --device cuda --run-name clip

# If OOM, use accumulation:
# !python -m ai.training.train_clip --epochs 5 --batch-size 8 --grad-accum 4 --device cuda

In [ ]:
from pathlib import Path
import json
metrics = json.load(open('models/checkpoints/clip/metrics.json'))
print('best_epoch', metrics.get('best_epoch'))
print('best_val_macro_f1', metrics.get('best_val_macro_f1'))
print('test', metrics.get('test_metrics'))
print('checkpoint', Path('models/checkpoints/clip/best_model.pt').resolve())
print('report', Path('runs/clip_training/report.md').resolve())

After training, download `models/checkpoints/clip/` into your local repo.
Keep `CLIP_FINETUNED_ENABLED=false` until you review metrics, then set it to `true`.